# Full Pipeline: Custom EHR → OMOP CDM v5.4

End-to-end standardization of a messy hospital EHR export to OMOP —
ingest → schema mapping → concept mapping → ETL → validation —
**fully offline**.

> **Prerequisites** — `pip install "portiere-health[polars]"`. This notebook is
> **fully offline**: no model downloads, no network calls, no API keys. It uses
> the bundled ICD-10-CM / LOINC / RxNorm vocabulary subset and BM25 lexical
> retrieval, and asserts the no-egress guarantee with `offline=True`.
> Every cell is deterministic — run top to bottom.

In [1]:
import json
import tempfile
from pathlib import Path

import polars as pl

import portiere
from portiere import __version__

WORK = Path(tempfile.mkdtemp(prefix="portiere_nb_"))
print(f"portiere v{__version__} — working dir: {WORK}")

portiere v0.4.0 — working dir: /var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_iv1omcob


## Step 1 — The source: a messy custom EHR export

Real hospital exports never match a standard. We generate a deliberately messy
one — cryptic column names, string-typed everything, empty cells — the shape
Portiere is built to untangle. In your project this is the extract your data
owner hands you.

In [2]:
import random

random.seed(42)
src_dir = WORK / "ehr_export"
src_dir.mkdir(parents=True)

n = 60
patients = pl.DataFrame({
    "patient_id":     [f"P{i:05d}" for i in range(1, n + 1)],
    "sex":            [random.choice(["M", "F", "F", "M", "U"]) for _ in range(n)],
    "date_of_birth":  [f"{random.randint(1940, 2005)}-{random.randint(1, 12):02d}-15"
                       for _ in range(n)],
    "race":           [random.choice(["CHINESE", "MALAY", "INDIAN", ""]) for _ in range(n)],
})
patients.write_csv(src_dir / "patients.csv")

dx_codes = ["E11.9", "I10", "J06.9", "K21.0", "E78.5"]
diagnoses = pl.DataFrame({
    "patient_id":       [f"P{random.randint(1, n):05d}" for _ in range(n * 2)],
    "diagnosis_code":   [random.choice(dx_codes) for _ in range(n * 2)],
    "diagnosis_date":   [f"2024-{random.randint(1, 12):02d}-{random.randint(1, 28):02d}"
                         for _ in range(n * 2)],
    "diagnosis_description": ["" for _ in range(n * 2)],
})
diagnoses.write_csv(src_dir / "diagnoses.csv")
print(patients.head(3))
print(diagnoses.head(3))

shape: (3, 4)
┌────────────┬─────┬───────────────┬────────┐
│ patient_id ┆ sex ┆ date_of_birth ┆ race   │
│ ---        ┆ --- ┆ ---           ┆ ---    │
│ str        ┆ str ┆ str           ┆ str    │
╞════════════╪═════╪═══════════════╪════════╡
│ P00001     ┆ M   ┆ 1969-02-15    ┆ MALAY  │
│ P00002     ┆ M   ┆ 1988-05-15    ┆ INDIAN │
│ P00003     ┆ F   ┆ 1998-11-15    ┆        │
└────────────┴─────┴───────────────┴────────┘
shape: (3, 4)
┌────────────┬────────────────┬────────────────┬───────────────────────┐
│ patient_id ┆ diagnosis_code ┆ diagnosis_date ┆ diagnosis_description │
│ ---        ┆ ---            ┆ ---            ┆ ---                   │
│ str        ┆ str            ┆ str            ┆ str                   │
╞════════════╪════════════════╪════════════════╪═══════════════════════╡
│ P00029     ┆ J06.9          ┆ 2024-07-19     ┆                       │
│ P00052     ┆ E11.9          ┆ 2024-04-09     ┆                       │
│ P00056     ┆ I10            ┆ 2024-01-23     

## Step 2 — Offline project configuration

The knowledge layer is a BM25 index built from the **bundled** Athena-format
vocabulary subset (ICD-10-CM, LOINC, RxNorm — license-clean to ship). For
production you build the same index from your own Athena export
(`docs/documentations/15-vocabulary-setup.md` covers SNOMED licensing).

In [3]:
# Offline configuration: BM25 retrieval over the bundled vocabulary subset,
# no embedding model (nothing to download), no reranker, and offline=True —
# construction FAILS if any provider could send data off this machine.
from portiere._demo_data import vocabulary_dir
from portiere.config import (
    EmbeddingConfig,
    KnowledgeLayerConfig,
    PortiereConfig,
    RerankerConfig,
)
from portiere.knowledge import build_knowledge_layer

knowledge_paths = build_knowledge_layer(
    athena_path=str(vocabulary_dir()),
    output_path=str(WORK / "knowledge_index"),
    backend="bm25s",
    vocabularies=["ICD10CM", "LOINC", "RxNorm"],
)

config = PortiereConfig(
    local_project_dir=WORK / "project",
    knowledge_layer=KnowledgeLayerConfig(backend="bm25s", **knowledge_paths),
    embedding=EmbeddingConfig(provider="none"),
    reranker=RerankerConfig(provider="none", model=""),
    offline=True,  # the executable no-egress guarantee (v0.4.0)
)
print("offline mode asserted:", config.offline)

2026-07-24 10:10:51 [info     ] athena.loading_concepts        path=/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/src/portiere/_demo_data/vocabulary/CONCEPT.csv


2026-07-24 10:10:51 [info     ] athena.concepts_loaded         count=27 vocabularies=['ICD10CM', 'LOINC', 'RxNorm']


2026-07-24 10:10:51 [info     ] athena.bm25s_built             count=27 path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_iv1omcob/knowledge_index/concepts.json


2026-07-24 10:10:51 [info     ] athena.knowledge_layer_built   backend=bm25s concepts=27 paths={'bm25s_corpus_path': '/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_iv1omcob/knowledge_index/concepts.json'}


offline mode asserted: True


In [4]:
project = portiere.init(
    name="ehr-to-omop",
    target_model="omop_cdm_v5.4",
    vocabularies=["ICD10CM", "LOINC", "RxNorm"],
    config=config,
)
print(project)

2026-07-24 10:10:51 [info     ] PolarsEngine initialized      


2026-07-24 10:10:51 [debug    ] local_storage.initialized      base_dir=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_iv1omcob/project


2026-07-24 10:10:51 [info     ] local_storage.project_created  name=ehr-to-omop project_id=48f6c6ac-ec84-47ec-b317-07c302f7f056


Project(name='ehr-to-omop', task='standardize', target_model='omop_cdm_v5.4', mode='local')


## Step 3 — Stage 1: ingest & profile

`add_source` profiles the file: column types, cardinality, missingness, plus
detected **code columns** (candidates for concept mapping) and **PHI columns**
(name-pattern heuristic; see `docs/phi-scrubbing.md` for value-level
scrubbing).

In [5]:
src_dx = project.add_source(str(src_dir / "diagnoses.csv"), name="diagnoses")
cols = [c["name"] if isinstance(c, dict) else c for c in src_dx["columns"]]
print(f"rows: {src_dx['row_count']}, columns: {len(cols)}")
print("columns:", cols)

2026-07-24 10:10:51 [info     ] project.source_added           project=ehr-to-omop source=diagnoses


2026-07-24 10:10:51 [warning  ] manifest.git_dirty             hint='Working tree has uncommitted changes; manifest records git_dirty=true' project_root=/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/docs/notebooks_examples


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-07-24 10:10:55 [info     ] gx_profiler.profiled           columns=4 rows=120 source=diagnoses


2026-07-24 10:10:55 [info     ] project.profiled               source=diagnoses


rows: 120, columns: 4
columns: ['patient_id', 'diagnosis_code', 'diagnosis_date', 'diagnosis_description']


## Step 4 — Stage 2: schema mapping

Source columns → OMOP tables/fields via pattern matching + lexical retrieval.
Every mapping carries a confidence and a routing status — nothing is silently
auto-applied below threshold.

In [6]:
schema_map = project.map_schema(src_dx)
for item in schema_map.items:
    print(f"  {item.source_column:12s} -> "
          f"{(item.effective_target_table or '?'):22s}."
          f"{(item.effective_target_column or '?'):26s} "
          f"conf={item.confidence:.2f} [{item.status}]")

2026-07-24 10:10:55 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=none target_model=omop_cdm_v5.4


2026-07-24 10:10:55 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=none target_standard=omop_cdm_v5.4


2026-07-24 10:10:55 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=none


2026-07-24 10:10:55 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=137


2026-07-24 10:10:55 [info     ] local_schema_mapper.initialized standard=omop_cdm_v5.4


2026-07-24 10:10:55 [info     ] Stage 2 complete               auto=3 review=0 total=4


2026-07-24 10:10:55 [info     ] local_storage.schema_mapping_saved items_count=4 project=ehr-to-omop


2026-07-24 10:10:55 [info     ] project.schema_mapped          auto=3 project=ehr-to-omop total=4


  patient_id   -> person                .person_id                  conf=0.95 [MappingStatus.AUTO_ACCEPTED]
  diagnosis_code -> condition_occurrence  .condition_source_value     conf=0.95 [MappingStatus.AUTO_ACCEPTED]
  diagnosis_date -> condition_occurrence  .condition_start_date       conf=0.95 [MappingStatus.AUTO_ACCEPTED]
  diagnosis_description -> condition_occurrence  .condition_source_value     conf=0.50 [MappingStatus.UNMAPPED]


## Step 5 — Stage 3: concept mapping

Local ICD-10 codes → standard concepts through the knowledge layer, with
confidence routing (auto / review / manual). `code_columns` is explicit here
for determinism; omit it to let Stage-1 auto-detection choose.

In [7]:
concept_map = project.map_concepts(source=src_dx, code_columns=["diagnosis_code"])
for item in concept_map.items:
    method = item.method.value if hasattr(item.method, "value") else item.method
    print(f"  {item.source_code:8s} -> {str(item.target_concept_id):10s} "
          f"{(item.target_concept_name or ''):40s} "
          f"conf={item.confidence:.2f} [{method}]")

2026-07-24 10:10:55 [info     ] Stage 3: Concept mapping (local) columns=['diagnosis_code'] knowledge_backend=bm25s


2026-07-24 10:10:55 [info     ] Mapping column: diagnosis_code


2026-07-24 10:10:55 [info     ] Found description column: diagnosis_description code_column=diagnosis_code desc_column=diagnosis_description mappings=5


2026-07-24 10:10:55 [info     ] knowledge.creating_backend     backend=bm25s path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_iv1omcob/knowledge_index/concepts.json


Split strings:   0%|          | 0/27 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-24 10:10:55 [info     ] Stage 3 complete               auto_rate=0.0% total_codes=5


2026-07-24 10:10:55 [info     ] local_storage.concept_mapping_saved items_count=5 project=ehr-to-omop


2026-07-24 10:10:55 [info     ] project.concepts_mapped        auto_rate=0.0 project=ehr-to-omop total=5


  J06.9    -> None                                                conf=0.00 [manual]
  I10      -> None                                                conf=0.00 [manual]
  E11.9    -> None                                                conf=0.00 [manual]
  E78.5    -> None                                                conf=0.00 [manual]
  K21.0    -> None                                                conf=0.00 [manual]


## Step 6 — Stage 4: ETL

Generates and runs the transform. The emitted artifacts (scripts + lookup
CSVs) run **without** Portiere installed — no runtime lock-in.

In [8]:
etl_out = WORK / "omop_output"
result = project.run_etl(src_dx, output_dir=str(etl_out),
                         schema_mapping=schema_map, concept_mapping=concept_map)
for f in sorted(etl_out.rglob("*.csv")):
    print(f"  {f.relative_to(etl_out)}  ({f.stat().st_size} bytes)")

2026-07-24 10:10:55 [info     ] Reading source                 format=csv path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_iv1omcob/ehr_export/diagnoses.csv


2026-07-24 10:10:55 [info     ] Processing table               columns=1 table=person


2026-07-24 10:10:55 [info     ] Processing table               columns=2 table=condition_occurrence


2026-07-24 10:10:55 [info     ] ETL complete                   duration=0.003437 success=True


2026-07-24 10:10:55 [info     ] project.etl_complete           output_dir=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_iv1omcob/omop_output project=ehr-to-omop


  condition_occurrence.csv  (2032 bytes)
  person.csv  (850 bytes)


In [9]:
# Inspect the OMOP condition_occurrence output
cond_files = sorted(etl_out.rglob("condition_occurrence*.csv"))
if cond_files:
    print(pl.read_csv(cond_files[0], infer_schema_length=0).head(5))
else:
    print("tables written:", [f.name for f in sorted(etl_out.rglob('*.csv'))])

shape: (5, 2)
┌────────────────────────┬──────────────────────┐
│ condition_source_value ┆ condition_start_date │
│ ---                    ┆ ---                  │
│ str                    ┆ str                  │
╞════════════════════════╪══════════════════════╡
│ J06.9                  ┆ 2024-07-19           │
│ E11.9                  ┆ 2024-04-09           │
│ I10                    ┆ 2024-01-23           │
│ J06.9                  ┆ 2024-07-01           │
│ J06.9                  ┆ 2024-09-26           │
└────────────────────────┴──────────────────────┘


## Step 7 — Stage 5: validate (optional extra)

Great-Expectations-backed completeness / conformance / plausibility scoring.
Guarded so the notebook also runs without the `[quality]` extra installed.

In [10]:
try:
    report = project.validate(output_path=str(etl_out))
    print(json.dumps({k: v for k, v in report.items()
                      if not isinstance(v, dict)}, indent=2, default=str))
except ImportError:
    print("Install portiere-health[quality] for Stage-5 validation — skipping.")
except Exception as exc:
    print(f"validation skipped: {type(exc).__name__}: {exc}")

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

2026-07-24 10:10:56 [info     ] gx_validator.validated         completeness=0.29 conformance=1.00 n_rules=3 overall_success=0.38 passed=False plausibility=1.00 table=condition_occurrence


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-07-24 10:10:56 [info     ] gx_validator.validated         completeness=0.11 conformance=1.00 n_rules=4 overall_success=0.11 passed=False plausibility=1.00 table=person


2026-07-24 10:10:56 [info     ] project.validated              all_passed=False project=ehr-to-omop tables=2


{
  "tables": [
    {
      "table_name": "condition_occurrence",
      "passed": false,
      "completeness_score": 0.2857142857142857,
      "conformance_score": 1.0,
      "plausibility_score": 1.0,
      "overall_success_score": 0.375,
      "plausibility_rule_results": [
        {
          "rule_id": "condition_concept_fk",
          "severity": "error",
          "passed": true,
          "total_rows": 0,
          "failed_count": 0,
          "detail": "column 'condition_concept_id' not in DataFrame (skipped)"
        },
        {
          "rule_id": "condition_start_before_end",
          "severity": "error",
          "passed": true,
          "total_rows": 0,
          "failed_count": 0,
          "detail": "column 'condition_end_date' not in DataFrame (skipped)"
        },
        {
          "rule_id": "condition_dates_consistent",
          "severity": "warn",
          "passed": true,
          "total_rows": 0,
          "failed_count": 0,
          "detail": "condition

## Step 8 — Deliverables & reproducibility

Every run emits a `manifest.lock.json` (model identity, vocab fingerprints,
thresholds — replayable via `portiere replay`). For the stakeholder-facing
Excel deliverables (scan workbook, schema working doc, concept map), see
`portiere workbook` in `docs/workbooks.md`.

In [11]:
manifests = sorted((WORK / "project").rglob("manifest.lock.json"))
print("manifest:", manifests[-1].name if manifests else "(none)")

# The same pipeline as reviewable Excel deliverables:
from portiere.models.schema_mapping import SchemaMapping  # noqa: F401
try:
    from portiere.reports import build_schema_workbook
    wb_path = build_schema_workbook(schema_map, WORK / "schema_working.xlsx",
                                    standard="omop_cdm_v5.4")
    print("schema working workbook:", wb_path.name)
except ImportError:
    print('Install portiere-health[xlsx] for Excel deliverables — skipping.')

manifest: (none)


schema working workbook: schema_working.xlsx
